In [ ]:
#part one

import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# ==========================================
# STEP 1: CREATE ENTERPRISE SUPPORT DATASET
# ==========================================
data = {
    'ticket_text': [
        "I cannot log in to my account, password reset link is broken",
        "Where can I download my annual tax form and salary slip?",
        "My payment failed twice but money was deducted from my credit card",
        "The application crashes every time I upload a large PDF file",
        "I need to change my direct deposit banking details for next month",
        "Billing error on my invoice, charged $200 instead of $100",
        "System throwing HTTP 500 internal server error during login",
        "Requesting update on my parental leave request status",
        "Subscription renewal failed due to expired credit card details",
        "Cannot view performance appraisal form on the employee portal",
        "Overcharged on my monthly software license payment",
        "Login screen is completely frozen on mobile app"
    ],
    'category': [
        'Account Access', 'HR & Admin', 'Billing', 'Technical',
        'HR & Admin', 'Billing', 'Technical', 'HR & Admin',
        'Billing', 'HR & Admin', 'Billing', 'Account Access'
    ]
}

# Expand synthetic dataset to simulate production volume
df = pd.DataFrame(data)
df = pd.concat([df] * 100, ignore_index=True)  # 1,200 samples

# Split into Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(
    df['ticket_text'], df['category'], test_size=0.2, random_state=42, stratify=df['category']
)

print(f"Dataset Loaded: {len(X_train)} training tickets, {len(X_test)} test tickets.")


# ==========================================
# STEP 2: BUILD PRODUCTION ML PIPELINE
# ==========================================
# - TfidfVectorizer converts raw text into numerical log-weighted features
# - MultinomialNB handles sparse text frequency vectors with Laplace smoothing (alpha=1.0)
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),        # Capture single words and 2-word pairs
        stop_words='english',      # Remove non-informative words ('the', 'is')
        max_features=5000          # Limit vocabulary size to prevent memory bloat
    )),
    ('classifier', MultinomialNB(alpha=1.0)) # Laplace smoothing enabled
])

# Fit pipeline on training data
train_start = time.time()
pipeline.fit(X_train, y_train)
train_end = time.time()

print(f"Model Training Complete in {(train_end - train_start) * 1000:.2f} ms")

# ==========================================
# STEP 3: OFFLINE EVALUATION
# ==========================================
y_pred = pipeline.predict(X_test)

print("\n--- Model Performance Report ---")
print(classification_report(y_test, y_pred))


# ==========================================
# STEP 4: REAL-TIME INFERENCE ENGINE
# ==========================================
def classify_incoming_ticket(ticket_text: str):
    """
    Simulates a microservice API endpoint receiving a raw text string.
    Returns predicted category, confidence score, and execution latency.
    """
    start_time = time.perf_counter()

    # 1. Predict class
    prediction = pipeline.predict([ticket_text])[0]

    # 2. Get prediction probabilities across all classes
    probabilities = pipeline.predict_proba([ticket_text])[0]
    confidence = np.max(probabilities)

    latency_ms = (time.perf_counter() - start_time) * 1000

    return {
        "ticket": ticket_text,
        "predicted_category": prediction,
        "confidence": round(float(confidence), 4),
        "latency_ms": round(latency_ms, 3)
    }

# --- Testing Real-time API Service with Unseen Tickets ---
incoming_tickets = [
    "I was billed twice for my annual license renewal yesterday",
    "How do I request my salary payslip for last month?",
    "App throws a 502 Bad Gateway error when clicking submit"
]

print("\n--- Live Production API Simulation ---")
for t in incoming_tickets:
    result = classify_incoming_ticket(t)
    print(f"Input:       \"{result['ticket']}\"")
    print(f"Routing To:  [{result['predicted_category']}] (Confidence: {result['confidence'] * 100:.1f}%)")
    print(f"Latency:     {result['latency_ms']} ms\n")

Dataset Loaded: 960 training tickets, 240 test tickets.
Model Training Complete in 69.67 ms

--- Model Performance Report ---
                precision    recall  f1-score   support

Account Access       1.00      1.00      1.00        40
       Billing       1.00      1.00      1.00        80
    HR & Admin       1.00      1.00      1.00        80
     Technical       1.00      1.00      1.00        40

      accuracy                           1.00       240
     macro avg       1.00      1.00      1.00       240
  weighted avg       1.00      1.00      1.00       240


--- Live Production API Simulation ---
Input:       "I was billed twice for my annual license renewal yesterday"
Routing To:  [Billing] (Confidence: 93.9%)
Latency:     6.256 ms

Input:       "How do I request my salary payslip for last month?"
Routing To:  [HR & Admin] (Confidence: 98.7%)
Latency:     9.697 ms

Input:       "App throws a 502 Bad Gateway error when clicking submit"
Routing To:  [Account Access] (Confid

In [ ]:
# ==============================================================================
# UNIVERSAL NLP SUPPORT ROUTER
# Industry Example: Banking Helpdesk Automation
# ==============================================================================

import time
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# ------------------------------------------------------------------------------
# STEP 1: PREPARE THE DATASET
# ------------------------------------------------------------------------------
# We create a dictionary of sample customer messages mapped to their correct category.
raw_data = {
    'ticket_text': [
        # 1. Dispute Category
        "My card was charged twice for the same transaction",
        "Unauthorized transaction on my debit card, I did not make this payment",
        "Someone used my credit card for online shopping, I want a refund",
        "Double deduction happened on my account yesterday",
        "Fraudulent UPI transaction, please reverse the money",
        # 2. New Account Category
        "I want to open a savings account, what documents are needed",
        "How do I apply for a new current account for my business",
        "Interested in opening a fixed deposit account, please guide me",
        "Can I open a joint account with my spouse online",
        "What is the process for a new NRI account",
        # 3. Account Access Category
        "Reset my net banking password, I forgot it",
        "Cannot login to mobile banking app, OTP not received",
        "My internet banking account is locked, please unlock",
        "Forgot my ATM PIN, how to regenerate it",
        "Unable to access my account after changing phone number",
        # 4. Loan Category
        "What is the interest rate for a home loan",
        "My personal loan application status, please update",
        "How to prepay my car loan without penalty",
        "Education loan eligibility criteria for masters abroad",
        "Home loan EMI not deducted this month, why",
        # 5. Card Services Category
        "My debit card is expiring next month, how to renew",
        "Request for a new credit card with higher limit",
        "My card was blocked, please unblock it",
        "How to enable international transactions on my card",
        "Lost my credit card, need to report and get replacement",
    ],
    'category': [
        'Dispute',
        'Dispute',
        'Dispute',
        'Dispute',
        'Dispute',
        'New Account',
        'New Account',
        'New Account',
        'New Account',
        'New Account',
        'Account Access',
        'Account Access',
        'Account Access',
        'Account Access',
        'Account Access',
        'Loan',
        'Loan',
        'Loan',
        'Loan',
        'Loan',
        'Card Services',
        'Card Services',
        'Card Services',
        'Card Services',
        'Card Services',
    ],
}

# Convert dictionary into a pandas DataFrame (spreadsheet-like table)
df = pd.DataFrame(raw_data)

# Duplicate dataset 40x to simulate 1,000 incoming tickets for realistic training
df = pd.concat([df] * 40, ignore_index=True)

# Split dataset: 80% used for teaching the model, 20% reserved for testing its accuracy
X_train, X_test, y_train, y_test = train_test_split(
    df['ticket_text'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category'],
)

print(f"[BANK] Dataset Ready: {len(X_train)} training examples, {len(X_test)} testing examples.")

# ------------------------------------------------------------------------------
# STEP 2: BUILD & TRAIN THE MACHINE LEARNING PIPELINE
# ------------------------------------------------------------------------------
# The Pipeline combines text conversion (TF-IDF) and learning (Naive Bayes) into 1 process
pipeline = Pipeline([
    # Convert text to numbers, using single words & 2-word pairs (ngrams), excluding stop words
    (
        'tfidf',
        TfidfVectorizer(
            ngram_range=(1, 2), stop_words='english', max_features=5000
        ),
    ),
    # Naive Bayes classifier: calculates probabilities of categories based on words used
    ('classifier', MultinomialNB(alpha=1.0)),
])

# Train the model and measure training speed
start_train = time.time()
pipeline.fit(X_train, y_train)
train_time_ms = (time.time() - start_train) * 1000
print(f"[BANK] Model trained successfully in {train_time_ms:.2f} ms!")

# ------------------------------------------------------------------------------
# STEP 3: EVALUATE MODEL ACCURACY
# ------------------------------------------------------------------------------
# Make predictions on the hidden 20% test dataset
y_pred = pipeline.predict(X_test)

print("\n--- Model Accuracy Metrics ---")
print(classification_report(y_test, y_pred))

# ------------------------------------------------------------------------------
# STEP 4: SAVE TRAINED MODEL TO DISK
# ------------------------------------------------------------------------------
# Save the model into a binary .pkl file so it can be loaded into an API server without retraining
model_filename = "banking_router_v1.pkl"
joblib.dump(pipeline, model_filename)
print(f"[BANK] Model successfully saved as '{model_filename}'")

# ------------------------------------------------------------------------------
# STEP 5: REAL-TIME API INFERENCE FUNCTION
# ------------------------------------------------------------------------------
def classify_incoming_ticket(ticket_text: str):
    """Simulates receiving a ticket via API, returning predicted department, confidence, and response speed."""
    start_time = time.perf_counter()

    # Predict category and calculate probability for all classes
    prediction = pipeline.predict([ticket_text])[0]
    probabilities = pipeline.predict_proba([ticket_text])[0]
    confidence_score = float(np.max(probabilities))

    # Calculate end-to-end execution speed in milliseconds
    latency_ms = (time.perf_counter() - start_time) * 1000

    return {
        "ticket": ticket_text,
        "predicted_category": prediction,
        "confidence": round(confidence_score, 4),
        "latency_ms": round(latency_ms, 3),
    }


# ------------------------------------------------------------------------------
# STEP 6: SIMULATE LIVE PRODUCTION TICKETS
# ------------------------------------------------------------------------------
live_incoming_tickets = [
    "My card was charged twice at Amazon yesterday",
    "I want to open a new savings account",
    "Reset my net banking password please",
    "What is the home loan interest rate currently",
    "Lost my debit card, need to block it urgently",
]

print("\n--- Live Real-Time API Simulation ---")
for ticket in live_incoming_tickets:
    result = classify_incoming_ticket(ticket)
    print(f"Input Text: \"{result['ticket']}\"")
    print(
        f"Assigned To: [{result['predicted_category']}]"
        f" ({result['confidence']*100:.1f}% confidence)"
    )
    print(f"Latency:     {result['latency_ms']} ms\n")



'''
🌐 In a Real Production System
The incoming_tickets list would be replaced by actual HTTP requests:

python
# Real-world equivalent of Step 4
@app.post("/classify")
def classify(ticket: dict):
    # ticket = {"text": "I was billed twice..."}  ← from the user's POST
    return classify_incoming_ticket(ticket["text"])
Every time a support ticket is submitted via web form / email / chat → the API sends the text → model classifies it → response routes it to the right team.


'''

[BANK] Dataset Ready: 800 training examples, 200 testing examples.
[BANK] Model trained successfully in 16.90 ms!

--- Model Accuracy Metrics ---
                precision    recall  f1-score   support

Account Access       1.00      1.00      1.00        40
 Card Services       1.00      1.00      1.00        40
       Dispute       1.00      1.00      1.00        40
          Loan       1.00      1.00      1.00        40
   New Account       1.00      1.00      1.00        40

      accuracy                           1.00       200
     macro avg       1.00      1.00      1.00       200
  weighted avg       1.00      1.00      1.00       200

[BANK] Model successfully saved as 'banking_router_v1.pkl'

--- Live Real-Time API Simulation ---
Input Text: "My card was charged twice at Amazon yesterday"
Assigned To: [Dispute] (99.1% confidence)
Latency:     3.292 ms

Input Text: "I want to open a new savings account"
Assigned To: [New Account] (99.0% confidence)
Latency:     2.272 ms

Inpu

'\n🌐 In a Real Production System\nThe incoming_tickets list would be replaced by actual HTTP requests:\n\npython\n# Real-world equivalent of Step 4\n@app.post("/classify")\ndef classify(ticket: dict):\n    # ticket = {"text": "I was billed twice..."}  ← from the user\'s POST\n    return classify_incoming_ticket(ticket["text"])\nEvery time a support ticket is submitted via web form / email / chat → the API sends the text → model classifies it → response routes it to the right team.\n\n\n'